<a href="https://colab.research.google.com/github/smansipillai009/FloodLens/blob/main/FloodLens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/FloodLens && cd /content/FloodLens && unzip -q -o /content/drive/MyDrive/floodlens_code.zip
!mkdir -p /content/FloodLens/data/processed && cd /content/FloodLens/data/processed && unzip -q -o /content/drive/MyDrive/kurosiwo_patches.zip

Mounted at /content/drive


In [ ]:
!pip install -q segmentation-models-pytorch pyyaml
%cd /content/FloodLens
!python -c "import torch; print('GPU:', torch.cuda.is_available())"
!ls data/processed/kurosiwo'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 7.6 MB/s eta 0:00:00
/content/FloodLens
GPU: True
/bin/bash: -c: line 1: unexpected EOF while looking for matching `''


In [ ]:
%%writefile -a /content/FloodLens/app/ml/dataset.py


# --- Colab fix: flipped views have negative strides; force fresh contiguous copies ---
_orig_getitem = PatchDataset.__getitem__
def _safe_getitem(self, i):
    return tuple(np.ascontiguousarray(a).copy() for a in _orig_getitem(self, i))
PatchDataset.__getitem__ = _safe_getitem

Appending to /content/FloodLens/app/ml/dataset.py


In [ ]:
%%writefile /content/FloodLens/app/ml/dataset.py
"""Patch dataset for flood segmentation (accepts (4,H,W) or (4,1,H,W) patches)."""
from __future__ import annotations

from pathlib import Path

import numpy as np

from app.ml.model import make_input


class PatchDataset:
    def __init__(self, root, split, norm=None, augment=False):
        self.files = sorted((Path(root) / split).rglob("*.npz"))
        self.norm, self.augment = norm, augment
        if not self.files:
            raise FileNotFoundError(f"No .npz patches under {Path(root) / split}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        z = np.load(self.files[i])
        sar = z["sar"].astype("float32")
        h, w = sar.shape[-2:]
        sar = sar.reshape(4, h, w)
        x = make_input(*sar)
        if self.norm:
            x = (x - np.array(self.norm["mean"], "float32")[:, None, None]) / np.array(self.norm["std"], "float32")[:, None, None]
        y = z["y"].astype("float32").reshape(h, w)
        v = z["valid"].astype("float32").reshape(h, w)
        if self.augment:
            if np.random.rand() < .5: x, y, v = x[:, :, ::-1], y[:, ::-1], v[:, ::-1]
            if np.random.rand() < .5: x, y, v = x[:, ::-1], y[::-1], v[::-1]
        return (np.ascontiguousarray(x).copy(), np.ascontiguousarray(y).copy()[None],
                np.ascontiguousarray(v).copy()[None])


def compute_norm_stats(ds, max_patches=500):
    acc, acc2, n = 0.0, 0.0, 0
    for i in np.linspace(0, len(ds) - 1, min(len(ds), max_patches)).astype(int):
        x, _, v = ds[int(i)]
        m = v[0] > 0
        xs = x[:, m]
        acc, acc2, n = acc + xs.sum(1), acc2 + (xs ** 2).sum(1), n + xs.shape[1]
    mean = acc / n
    std = np.sqrt(np.maximum(acc2 / n - mean ** 2, 1e-6))
    return {"mean": mean.tolist(), "std": std.tolist()}

Overwriting /content/FloodLens/app/ml/dataset.py


In [ ]:
import os, sys, numpy as np
os.chdir('/content/FloodLens'); sys.path.insert(0, '/content/FloodLens')
from app.ml.dataset import PatchDataset

ds = PatchDataset('data/processed/kurosiwo', 'train')
idx = np.linspace(0, len(ds) - 1, 200).astype(int)
fl = [ds[int(i)][1].mean() for i in idx]
va = [ds[int(i)][2].mean() for i in idx]
print('x shape:', ds[0][0].shape, '| y shape:', ds[0][1].shape)
print('mean flood fraction:', float(np.mean(fl)))
print('mean valid fraction:', float(np.mean(va)))
print('patches with any flood:', int(sum(f > 0 for f in fl)), 'of 200')

x shape: (6, 224, 224) | y shape: (1, 224, 224)
mean flood fraction: 0.062193579971790314
mean valid fraction: 0.9353141784667969
patches with any flood: 174 of 200


In [ ]:
!python -m app.ml.train --data data/processed/kurosiwo --epochs 1 --max-train-patches 200

/content/FloodLens/app/ml/train.py:79: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  opt.zero_grad(); l = _loss(model(x), y, v); l.backward(); opt.step(); tl += float(l)
INFO:train:ep 0 loss 1.1851 val IoU 0.096 F1 0.175
{
  "val": {
    "iou": 0.09565510403272035,
    "f1": 0.17460805627728582,
    "precision": 0.09735397593448504,
    "recall": 0.8457152641041007,
    "area_diff_pct": 768.7013098193647,
    "inference_s_per_patch": 0.010793406387855267
  },
  "test_heldout_events": {
    "iou": 0.1356114736817106,
    "f1": 0.2388342788437162,
    "precision": 0.13597689041355837,
    "recall": 0.9805685658440807,
    "area_diff_pct": 621.1288350996939,
    "inference_s_per_patch": 0.012489562533067917
  },
  "ensemble_validated": false,
  "note": "Himalayan unseen-scene metrics are added by t

In [ ]:
!python -m app.ml.train --data data/processed/kurosiwo --epochs 15 --max-train-patches 4000

/content/FloodLens/app/ml/train.py:79: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  opt.zero_grad(); l = _loss(model(x), y, v); l.backward(); opt.step(); tl += float(l)
INFO:train:ep 0 loss 0.5314 val IoU 0.690 F1 0.817
INFO:train:ep 1 loss 0.2776 val IoU 0.700 F1 0.824
INFO:train:ep 2 loss 0.2522 val IoU 0.643 F1 0.783
INFO:train:ep 3 loss 0.2409 val IoU 0.686 F1 0.814
INFO:train:ep 4 loss 0.2239 val IoU 0.672 F1 0.804
INFO:train:ep 5 loss 0.2242 val IoU 0.710 F1 0.831
INFO:train:ep 6 loss 0.2116 val IoU 0.675 F1 0.806
INFO:train:ep 7 loss 0.1951 val IoU 0.711 F1 0.831
INFO:train:ep 8 loss 0.1927 val IoU 0.693 F1 0.819
INFO:train:ep 9 loss 0.1902 val IoU 0.659 F1 0.795
INFO:train:ep 10 loss 0.1994 val IoU 0.729 F1 0.844
INFO:train:ep 11 loss 0.1912 val IoU 0.656 F1 0.792
INFO:train:ep 12 loss 

In [ ]:
!mkdir -p /content/drive/MyDrive/floodlens_ckpt
!cp -r models/checkpoints/* /content/drive/MyDrive/floodlens_ckpt/
!cp data/processed/kurosiwo/prepare_meta.json /content/drive/MyDrive/floodlens_ckpt/
!ls -la /content/drive/MyDrive/floodlens_ckpt

total 56129
drwx------ 2 root root     4096 Oct 14 05:48 .
drwx------ 5 root root     4096 Oct 14 05:00 ..
-rw------- 1 root root     1371 Oct 14 05:48 baseline_tuning.json
-rw------- 1 root root      627 Oct 14 05:48 metrics.json
-rw------- 1 root root      255 Oct 14 05:48 norm_stats.json
-rw------- 1 root root      776 Oct 14 05:48 prepare_meta.json
-rw------- 1 root root 57463163 Oct 14 05:48 unet_resnet18.pt
